# Diabetic Retinopathy Model Verification Notebook

This notebook verifies loading the fine-tuned **InceptionV3** model (`best_inceptionv3_finetuned_S_93.h5`) and running predictions on fundus images.

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# 1. Load Model
MODEL_PATH = 'best_inceptionv3_finetuned_S_93.h5'
if not os.path.exists(MODEL_PATH):
    print(f"Error: {MODEL_PATH} not found in current directory!")
else:
    print(f"Loading model from {MODEL_PATH}...")
    model = keras.models.load_model(MODEL_PATH)
    print("Model loaded successfully!")
    print("Expected input shape:", model.input_shape)
    print("Output shape:", model.output_shape)

In [ ]:
# 2. Define DR Stage Labels and Normalization Function
DR_STAGES = {
    0: "No DR",
    1: "Mild DR",
    2: "Moderate DR",
    3: "Severe DR",
    4: "Proliferative DR"
}

def preprocess_image(image_input):
    """
    Accepts PIL Image, file path, or numpy array.
    Resizes to 299x299 (InceptionV3 input) and normalizes pixels to [-1, 1].
    """
    if isinstance(image_input, str):
        img = Image.open(image_input).convert('RGB')
    elif isinstance(image_input, np.ndarray):
        img = Image.fromarray(image_input.astype('uint8')).convert('RGB')
    else:
        img = image_input.convert('RGB')
        
    img_resized = img.resize((299, 299))
    arr = np.array(img_resized).astype('float32')
    # InceptionV3 preprocessing: [-1, 1]
    arr_normalized = (arr / 127.5) - 1.0
    # Add batch dimension: (1, 299, 299, 3)
    batch = np.expand_dims(arr_normalized, axis=0)
    return img, batch

In [ ]:
# 3. Run Test Prediction
def test_prediction(image_path=None):
    if image_path and os.path.exists(image_path):
        print(f"Testing image: {image_path}")
        display_img, batch_data = preprocess_image(image_path)
    else:
        print("No image path provided (or file not found). Generating synthetic test image (299x299)...")
        dummy_pixels = np.random.randint(0, 256, (299, 299, 3), dtype=np.uint8)
        display_img, batch_data = preprocess_image(dummy_pixels)
    
    # Run Inference
    preds = model.predict(batch_data)
    probabilities = preds[0]
    top_class_idx = int(np.argmax(probabilities))
    top_stage = DR_STAGES.get(top_class_idx, f"Class {top_class_idx}")
    confidence = float(probabilities[top_class_idx])
    
    # Plot Image & Class Probabilities
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(display_img)
    ax1.set_title(f"Predicted: {top_stage} ({confidence*100:.1f}%)")
    ax1.axis('off')
    
    stages_list = [DR_STAGES[i] for i in range(len(DR_STAGES))]
    ax2.barh(stages_list, probabilities, color='skyblue')
    ax2.set_xlabel('Probability')
    ax2.set_xlim([0, 1.0])
    ax2.set_title('DR Stage Probabilities')
    plt.tight_layout()
    plt.show()
    
    print(f"Predicted Stage: {top_stage} (Index {top_class_idx})")
    print(f"Confidence: {confidence:.4f}")
    for i, p in enumerate(probabilities):
        print(f"  - {DR_STAGES[i]}: {p:.4f}")

# Run test prediction (you can pass 'path/to/fundus_image.jpg')
test_prediction()